# 26 — Initialization and Gradient Flow

**Master syllabus coverage:** V2 2.13, 3.8

## Why this matters

A network can be mathematically valid yet untrainable because signals vanish, explode, saturate, or remain symmetric before optimization has a chance to help.

## Learning objectives

- Derive the fan-in variance heuristic.
- Compare bad, Xavier, and He initialization across depth.
- Explain and demonstrate symmetry breaking.
- Measure activation and gradient statistics layer by layer.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Initialization controls signal scale before learning

For independent zero-mean inputs and weights in $y_j=\sum_i w_{ji}x_i$,
$\operatorname{Var}(y_j)\approx fan_{in}\operatorname{Var}(w)\operatorname{Var}(x)$.
Choosing weight variance near $1/fan_{in}$ preserves forward variance for linear/tanh-like
regimes (Xavier). ReLU discards roughly half the signal, motivating variance near
$2/fan_{in}$ (He/Kaiming).


In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(42)
width, depth, batch = 256, 30, 512
initial = torch.randn(batch, width)

def propagate(std: float, activation) -> list[float]:
    x = initial.clone()
    variances = [x.var().item()]
    generator = torch.Generator().manual_seed(7)
    for _ in range(depth):
        weight = torch.randn(width, width, generator=generator) * std
        x = activation(x @ weight.T)
        variances.append(x.var().item())
    return variances

schemes = {
    "too small": propagate(0.01, torch.tanh),
    "too large": propagate(0.3, torch.tanh),
    "xavier": propagate(math.sqrt(1 / width), torch.tanh),
    "he + relu": propagate(math.sqrt(2 / width), F.relu),
}
for name, values in schemes.items():
    print(f"{name:10}: first={values[0]:.3f}, layer 10={values[10]:.3g}, final={values[-1]:.3g}")


## 2. Symmetry must be broken

If neurons in one layer start with identical weights and biases, they compute identical
outputs, receive identical gradients, and remain duplicates. Random initialization breaks
this symmetry. Zero biases are usually fine because random weights already distinguish units.


In [ ]:
x = torch.randn(8, 4)
shared_row = torch.randn(1, 4)
symmetric_weight = shared_row.repeat(3, 1).requires_grad_()
output = torch.tanh(x @ symmetric_weight.T)
loss = output.square().mean()
loss.backward()
assert torch.allclose(symmetric_weight.grad[0], symmetric_weight.grad[1])
print("identical neurons receive identical gradients:", symmetric_weight.grad[0])


## 3. Backward variance matters too

A deep product of Jacobians can shrink or explode. Activation derivatives, weight scale,
normalization, and residual paths all affect gradient flow. Inspect distributions across
depth—not only the final loss.


In [ ]:
def gradient_profile(init: str) -> list[float]:
    torch.manual_seed(3)
    layers = torch.nn.ModuleList([torch.nn.Linear(64, 64) for _ in range(20)])
    for layer in layers:
        if init == "small":
            torch.nn.init.normal_(layer.weight, std=0.01)
        elif init == "he":
            torch.nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        torch.nn.init.zeros_(layer.bias)
    x = torch.randn(32, 64)
    for layer in layers:
        x = F.relu(layer(x))
        x.retain_grad()
    x.square().mean().backward()
    return [layer.weight.grad.norm().item() for layer in layers]

for init in ("small", "he"):
    norms = gradient_profile(init)
    print(init, "first/middle/last weight grad norms:", norms[0], norms[10], norms[-1])


## 4. Framework initialization and measurement

Inspect actual initialized statistics rather than relying only on the function name.
Fan mode, nonlinearity gain, truncation, bias policy, and parameter orientation change
the expected scale.


In [ ]:
layer = torch.nn.Linear(128, 512, bias=False)
torch.nn.init.xavier_normal_(layer.weight)
expected_std = math.sqrt(2 / (128 + 512))
measured_std = layer.weight.std().item()
print("expected std:", expected_std, "measured std:", measured_std)
assert abs(measured_std - expected_std) / expected_std < 0.1


## Failure modes to recognize

- **Vanishing activations/gradients:** deeper layers receive almost no usable signal.
- **Exploding scale:** activations, loss, or gradients become huge or non-finite.
- **Symmetric neurons:** capacity is wasted on permanently identical units.
- **Wrong fan convention:** parameter orientation makes the selected scale inappropriate.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Add sigmoid to the depth experiment and measure saturation frequency.
2. Compare Xavier uniform with Xavier normal across ten seeds.
3. Record activation mean, standard deviation, and zero fraction for every ReLU layer.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can choose and justify an initializer, then verify its forward and backward statistics empirically.

**Next:** 27 — Stable depth and generalization.
